## SVM с масштабированием

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import SVC
from joblib import parallel_backend
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate

# Загрузка данных
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
X = train.drop(columns=["Id", "Survived"]).copy()
y = train["Survived"]
X_test = test.drop(columns="Id").copy()
for data in [X, X_test]:
    data["around_whealth"] = (data["Parch"] + data["SibSp"]) * data["Fare"]


## Подготовка признаков

In [2]:
# Преобразования внутри фолдов
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare", "around_whealth"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_columns),
    ("categorical", categorical_transformer, ["Sex", "Embarked"]),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC(kernel="rbf", random_state=42)),
])


## Подбор параметров

In [3]:
# Пятикратная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {"model__C": [0.1, 1, 10, 100], "model__gamma": ["scale", 0.001, 0.01, 0.1]}
search = GridSearchCV(
    pipeline, param_grid, scoring={"f1": "f1", "roc_auc": "roc_auc"},
    refit="f1", cv=cv, n_jobs=4, return_train_score=True,
)
with parallel_backend("threading"):
    search.fit(X, y)
print("Лучшие параметры:", search.best_params_)
print("F1 при подборе:", round(search.best_score_, 4))
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["params", "mean_train_f1", "mean_test_f1", "std_test_f1", "mean_test_roc_auc"]]
        .sort_values("mean_test_f1", ascending=False).head(10))


Лучшие параметры: {'model__C': 100, 'model__gamma': 0.001}
F1 при подборе: 0.7491


,params,mean_train_f1,mean_test_f1,std_test_f1,mean_test_roc_auc
13,"{'model__C': 100, 'model__gamma': 0.001}",0.774308,0.749147,0.057909,0.833751
6,"{'model__C': 1, 'model__gamma': 0.01}",0.764944,0.745398,0.058255,0.846565
8,"{'model__C': 10, 'model__gamma': 'scale'}",0.783599,0.745340,0.058259,0.832696
9,"{'model__C': 10, 'model__gamma': 0.001}",0.744341,0.742124,0.057105,0.842151
10,"{'model__C': 10, 'model__gamma': 0.01}",0.784886,0.741215,0.053824,0.835960
12,"{'model__C': 100, 'model__gamma': 'scale'}",0.793168,0.736574,0.049594,0.835749
4,"{'model__C': 1, 'model__gamma': 'scale'}",0.757581,0.736469,0.054236,0.854039
14,"{'model__C': 100, 'model__gamma': 0.01}",0.808923,0.725457,0.048528,0.838814
7,"{'model__C': 1, 'model__gamma': 0.1}",0.800360,0.723206,0.052822,0.835833
3,"{'model__C': 0.1, 'model__gamma': 0.1}",0.752855,0.695570,0.060128,0.833880


## Независимая оценка подбора

In [4]:
# Вложенная кросс-валидация
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
with parallel_backend("threading"):
    nested_scores = cross_validate(
        search, X, y, cv=outer_cv, scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=1,
    )
fold_results = pd.DataFrame({
    "Фолд": range(1, 6),
    "F1": nested_scores["test_f1"],
    "ROC AUC": nested_scores["test_roc_auc"],
})
display(fold_results)
print("F1:", round(fold_results["F1"].mean(), 4), "±", round(fold_results["F1"].std(ddof=0), 4))
print("ROC AUC:", round(fold_results["ROC AUC"].mean(), 4), "±", round(fold_results["ROC AUC"].std(ddof=0), 4))


,Фолд,F1,ROC AUC
0,1,0.757576,0.862846
1,2,0.759690,0.823396
2,3,0.759690,0.837166
3,4,0.748092,0.828409
4,5,0.738462,0.813655


F1: 0.7527 ± 0.0083
ROC AUC: 0.8331 ± 0.0167


## Предсказания теста

In [5]:
# Сохранение посылки
best_model = search.best_estimator_
submission = pd.DataFrame({"Id": test["Id"], "Survived": best_model.predict(X_test)})
sample = pd.read_csv("data/sample_submission.csv")
assert submission.columns.tolist() == sample.columns.tolist()
assert submission["Id"].equals(sample["Id"])
assert submission["Survived"].isin([0, 1]).all()
assert not submission.isna().any().any()
submission.to_csv("submission_svm_scaled_cv.csv", index=False)
print("Сохранён submission_svm_scaled_cv.csv")


Сохранён submission_svm_scaled_cv.csv
